In [1]:
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
# elif torch.backends.mps.is_available():
#     device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Using device:", device)


Using device: cuda


In [3]:
from pathlib import Path
import cdsapi

download_path = Path("./downloads/era5")
download_path.mkdir(parents=True, exist_ok=True)

c = cdsapi.Client()

# Static
static_file = download_path / "static.nc"
if not static_file.exists():
    c.retrieve(
        "reanalysis-era5-single-levels",
        {
            "product_type": "reanalysis",
            "variable": ["geopotential", "land_sea_mask", "soil_type"],
            "year": "2023",
            "month": "01",
            "day": "01",
            "time": "00:00",
            "format": "netcdf",
        },
        str(static_file),
    )
print("Static variables downloaded!")

# Surface (4 times only)
surf_file = download_path / "2023-01-01-surface-level.nc"
if not surf_file.exists():
    c.retrieve(
        "reanalysis-era5-single-levels",
        {
            "product_type": "reanalysis",
            "variable": [
                "2m_temperature",
                "10m_u_component_of_wind",
                "10m_v_component_of_wind",
                "mean_sea_level_pressure",
            ],
            "year": "2023",
            "month": "01",
            "day": "01",
            "time": ["00:00", "06:00", "12:00", "18:00"],
            "format": "netcdf",
        },
        str(surf_file),
    )
print("Surface-level variables downloaded!")

# Pressure levels (4 times only)
atm_file = download_path / "2023-01-01-atmospheric.nc"
if not atm_file.exists():
    c.retrieve(
        "reanalysis-era5-pressure-levels",
        {
            "product_type": "reanalysis",
            "variable": [
                "temperature",
                "u_component_of_wind",
                "v_component_of_wind",
                "specific_humidity",
                "geopotential",
            ],
            "pressure_level": ["50","100","150","200","250","300","400","500","600","700","850","925","1000"],
            "year": "2023",
            "month": "01",
            "day": "01",
            "time": ["00:00", "06:00", "12:00", "18:00"],
            "format": "netcdf",
        },
        str(atm_file),
    )
print("Atmospheric variables downloaded!")


Static variables downloaded!
Surface-level variables downloaded!
Atmospheric variables downloaded!


In [4]:
import xarray as xr
import torch
from aurora import Batch, Metadata

static_vars_ds = xr.open_dataset(static_file, engine="netcdf4")
surf_vars_ds   = xr.open_dataset(surf_file, engine="netcdf4")
atmos_vars_ds  = xr.open_dataset(atm_file, engine="netcdf4")

batch = Batch(
    surf_vars={
        "2t":  torch.from_numpy(surf_vars_ds["t2m"].values[:2][None]),
        "10u": torch.from_numpy(surf_vars_ds["u10"].values[:2][None]),
        "10v": torch.from_numpy(surf_vars_ds["v10"].values[:2][None]),
        "msl": torch.from_numpy(surf_vars_ds["msl"].values[:2][None]),
    },
    static_vars={
        "z":   torch.from_numpy(static_vars_ds["z"].values[0]),
        "slt": torch.from_numpy(static_vars_ds["slt"].values[0]),
        "lsm": torch.from_numpy(static_vars_ds["lsm"].values[0]),
    },
    atmos_vars={
        "t": torch.from_numpy(atmos_vars_ds["t"].values[:2][None]),
        "u": torch.from_numpy(atmos_vars_ds["u"].values[:2][None]),
        "v": torch.from_numpy(atmos_vars_ds["v"].values[:2][None]),
        "q": torch.from_numpy(atmos_vars_ds["q"].values[:2][None]),
        "z": torch.from_numpy(atmos_vars_ds["z"].values[:2][None]),
    },
    metadata=Metadata(
        lat=torch.from_numpy(surf_vars_ds.latitude.values),
        lon=torch.from_numpy(surf_vars_ds.longitude.values),
        time=(surf_vars_ds.valid_time.values.astype("datetime64[s]").tolist()[1],),
        atmos_levels=tuple(int(level) for level in atmos_vars_ds.pressure_level.values),
    ),
)


## Loading and Running the Model

Finally, we are ready to load and run the model and visualise the predictions. We perform a roll-out for two steps, which produces predictions for hours 12:00 and 18:00.

In [7]:
from aurora import AuroraSmallPretrained, rollout

model = AuroraSmallPretrained(use_lora=False)
model.load_checkpoint("microsoft/aurora", "aurora-0.25-small-pretrained.ckpt")

model.eval()
model = model.to(device)

with torch.inference_mode():
    preds = [pred.to("cpu") for pred in rollout(model, batch, steps=1)]

print("Got", len(preds), "prediction steps.")


Got 1 prediction steps.


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(2, 2, figsize=(12, 6.5))

for i in range(2):
    pred = preds[i]

    ax[i, 0].imshow(pred.surf_vars["2t"][0, 0].numpy() - 273.15, vmin=-50, vmax=50)
    ax[i, 0].set_ylabel(str(pred.metadata.time[0]))
    if i == 0:
        ax[i, 0].set_title("Aurora Prediction")
    ax[i, 0].set_xticks([]); ax[i, 0].set_yticks([])

    ax[i, 1].imshow(surf_vars_ds["t2m"][2 + i].values - 273.15, vmin=-50, vmax=50)
    if i == 0:
        ax[i, 1].set_title("ERA5")
    ax[i, 1].set_xticks([]); ax[i, 1].set_yticks([])

plt.tight_layout()
plt.show()


# now let's try to add precipitation

In [9]:
from pathlib import Path
import cdsapi

download_path = Path("./downloads/era5")
download_path.mkdir(parents=True, exist_ok=True)

c = cdsapi.Client()

tp_file = download_path / "2023-01-01-tp-hourly-01_to_06.nc"
if not tp_file.exists():
    c.retrieve(
        "reanalysis-era5-single-levels",
        {
            "product_type": "reanalysis",
            "variable": ["total_precipitation"],
            "year": "2023",
            "month": "01",
            "day": "01",
            "time": [f"{h:02d}:00" for h in range(1, 7)],  # 01..06 inclusive
            "format": "netcdf",
        },
        str(tp_file),
    )

print("TP hourly file:", tp_file)


2025-12-17 12:05:03,832 INFO [2025-12-03T00:00:00Z] To improve our C3S service, we need to hear from you! Please complete this very short [survey](https://confluence.ecmwf.int/x/E7uBEQ/). Thank you.
2025-12-17 12:05:04,412 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2025-12-17 12:05:04,413 INFO Request ID is a4e0ea43-06d7-4362-a3dd-d798c7e31db5
2025-12-17 12:05:04,571 INFO status has been updated to accepted
2025-12-17 12:05:55,025 INFO status has been updated to successful


1ac9b18a809b07006d9a6ca3b14155d9.nc:   0%|          | 0.00/5.33M [00:00<?, ?B/s]

TP hourly file: downloads\era5\2023-01-01-tp-hourly-01_to_06.nc


In [23]:
import xarray as xr
import numpy as np

tp_ds = xr.open_dataset(tp_file, engine="netcdf4")
tp = tp_ds["tp"]   # dims: valid_time, latitude, longitude

# simple sum is correct (you already downloaded 01–06 only)
tp6h_ending_06 = tp.sum(dim="valid_time")

tp6h_log = np.log1p(tp6h_ending_06)


In [11]:
tp6h_mm = tp6h_ending_06 * 1000.0
print("tp6h mean (mm):", float(tp6h_mm.mean()))
print("tp6h max  (mm):", float(tp6h_mm.max()))


tp6h mean (mm): 0.5518486499786377
tp6h max  (mm): 143.2919464111328


# step 3: Computing normalization stats

In [13]:
import numpy as np

vals = y.values.ravel()
vals = vals[~np.isnan(vals)]


In [14]:
mu_tp = float(vals.mean())
std_tp = float(vals.std())
eps = 1e-6
std_tp = max(std_tp, eps)


In [15]:
print(mu_tp)
print(std_tp)

0.0005497062811627984
0.0019855170976370573


In [16]:
#save them
import json

stats = {
    "target": "log1p_tp_6h",
    "mean": mu_tp,
    "std": std_tp,
    "units": "log(1 + meters)",
}

with open("tp_normalization.json", "w") as f:
    json.dump(stats, f, indent=2)


In [20]:
model.eval()
batch = batch.to(device)  # make sure batch is on same device as model

with torch.inference_mode():
    pred = model(batch)

print(pred.surf_vars.keys())
print(pred.surf_vars["2t"].shape)



dict_keys(['2t', '10u', '10v', 'msl'])
torch.Size([1, 1, 720, 1440])
